# Day 48 · One self-attention head in code

**First-principles question:** What are queries, keys and values in real code?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

## 1 · One head

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.last_weights = None

    def forward(self, x):
        B, T, C = x.shape
        # TODO: k = self.key(x); q = self.query(x)
        #       wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        #       wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf")); wei = F.softmax(wei, dim=-1)
        #       self.last_weights = wei
        #       return wei @ self.value(x)
        ...

head = Head(C, 16, T)
out = head(x)
print(out.shape)
print(head.last_weights[0])

## 2 · Why divide by √head_size?

In [ ]:
q = torch.randn(B, T, 16); k = torch.randn(B, T, 16)
raw = q @ k.transpose(-2, -1)
print(f"variance without scaling: {raw.var():.2f} · with: {(raw * 16 ** -0.5).var():.2f}")
print("softmax of big numbers:", F.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]) * 8, dim=-1))

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert out.shape == (B, T, 16)
w = head.last_weights
assert torch.allclose(w.sum(-1), torch.ones(B, T)), "Weights must add up to 1"
assert (w.triu(1) == 0).all(), "The future must be masked"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")